# Autoencoder CASIA v2.0 — Complete Faculty Demo

Run this notebook top-to-bottom in Google Colab. Select a GPU runtime only when retraining is required.

In [ ]:
# Faculty demonstrations should load saved results instead of retraining.
TRAIN_MODEL = False
REPO_URL = "https://github.com/chetanraje27/Digital-Evidence-GenAI.git"
DATASET_HANDLE = "divg07/casia-20-image-tampering-detection-dataset"
print("TRAIN_MODEL =", TRAIN_MODEL)

## 1 — Project Introduction

**Project:** Multi-Model Generative AI Framework for Digital Evidence Analysis and Intelligence Generation  
**Component demonstrated here:** Convolutional Autoencoder  
**Dataset:** CASIA v2.0 Image Tampering Detection Dataset

The Autoencoder objectives are image reconstruction, meaningful dimensionality reduction, a later denoising experiment, and exploratory comparison of reconstruction behavior for authentic and tampered images. Authentic/tampered labels are retained only for analysis. This model is **not** claimed to be an automatic forgery detector.

## 2 — Environment and GPU

In [ ]:
%pip install -q kagglehub Pillow matplotlib numpy pandas scikit-image

import os, sys, subprocess, platform
from pathlib import Path
import torch

PROJECT_ROOT = Path("/content/Digital-Evidence-GenAI") if Path("/content").exists() else Path.cwd().resolve()
if Path("/content").exists():
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available"
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", GPU_NAME)
print("Selected device:", DEVICE)

## 3 — Dataset Overview

The main AE dataset contains **12,614 RGB images**: 7,491 authentic and 5,123 tampered. The separate 5,123 ground-truth PNG masks are excluded from normal Autoencoder inputs.

In [ ]:
import kagglehub, matplotlib.pyplot as plt, pandas as pd
from IPython.display import display

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
if not (RAW_DATA_DIR / "CASIA2" / "Au").is_dir():
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    kagglehub.dataset_download(DATASET_HANDLE, output_dir=str(RAW_DATA_DIR))
dataset_counts = pd.DataFrame({"Class": ["Authentic", "Tampered"], "Images": [7491, 5123]})
display(dataset_counts)
ax = dataset_counts.plot.bar(x="Class", y="Images", legend=False, color=["#2878B5", "#E07A1F"], figsize=(7,4))
ax.set_title("CASIA v2.0 Autoencoder Input Distribution")
ax.set_ylabel("Number of images")
ax.bar_label(ax.containers[0])
plt.tight_layout(); plt.show()

## 4 — Sample Digital Evidence

Authentic and tampered examples are displayed with their preserved metadata labels.

In [ ]:
from PIL import Image
import csv

with (PROJECT_ROOT / "data/splits/train.csv").open(newline="", encoding="utf-8") as file:
    training_rows = list(csv.DictReader(file))
sample_rows = [r for r in training_rows if r["class_name"] == "authentic"][:4] + [r for r in training_rows if r["class_name"] == "tampered"][:4]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, row in zip(axes.flat, sample_rows):
    with Image.open(PROJECT_ROOT / row["image_path"]) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(row["class_name"].title())
    ax.axis("off")
fig.suptitle("CASIA Digital Evidence Samples"); plt.tight_layout(); plt.show()

## 5 — Data Preprocessing

Each image is safely opened with PIL, converted to RGB, resized to 128×128, converted to a channel-first PyTorch tensor, and scaled to [0,1]. ImageNet normalization is not used.

In [ ]:
from ae_dataset import create_ae_dataloaders

loaders = create_ae_dataloaders(PROJECT_ROOT / "data/splits", image_size=128, batch_size=32, num_workers=0, seed=42)
example_batch = next(iter(loaders["train"]))
print("Input tensor shape:", list(example_batch["image"].shape))
print("Pixel minimum:", float(example_batch["image"].min()))
print("Pixel maximum:", float(example_batch["image"].max()))

## 6 — Data Splitting

The fixed seed is 42. Splits are reproducible and stratified, with zero duplicate leakage and zero ground-truth mask leakage.

In [ ]:
import json
pipeline_summary = json.loads((PROJECT_ROOT / "results/ae_data_pipeline_summary.json").read_text())
split_table = pd.DataFrame([
    {"Split": name.title(), "Total": pipeline_summary["split_counts"][name], **pipeline_summary["class_counts"][name]}
    for name in ("train", "validation", "test")
]).rename(columns={"authentic": "Authentic", "tampered": "Tampered"})
display(split_table)
print("Duplicate leakage:", pipeline_summary["duplicate_count_across_splits"])
print("Ground-truth mask leakage:", pipeline_summary["ground_truth_mask_count_accidentally_included"])

## 7 — Autoencoder Architecture

The encoder progressively reduces spatial dimensions to a compact latent representation. The decoder mirrors that process to reconstruct the original RGB image.

In [ ]:
from autoencoder import ConvolutionalAutoencoder
model = ConvolutionalAutoencoder().to(DEVICE)
architecture_table = pd.DataFrame([
    ["Input", "3 × 128 × 128", "49,152"],
    ["Encoder 1", "32 × 64 × 64", "—"],
    ["Encoder 2", "64 × 32 × 32", "—"],
    ["Encoder 3", "64 × 16 × 16", "—"],
    ["Latent", "32 × 8 × 8", "2,048"],
    ["Output", "3 × 128 × 128", "49,152"],
], columns=["Stage", "Shape", "Values per image"])
display(architecture_table)
print("Compression ratio: 24×")
print("Parameters:", sum(p.numel() for p in model.parameters()))

## 8 — Training Configuration

In [ ]:
display(pd.DataFrame({"Setting": ["Loss", "Optimizer", "Learning rate", "Batch size", "Maximum epochs", "Early-stopping patience"], "Value": ["MSELoss", "Adam", 0.001, 32, 20, 4]}))

## 9 — Training Control

`TRAIN_MODEL=False` loads the existing epoch-20 checkpoint for a quick faculty demonstration. Set it to `True` only when a GPU retraining run is intended.

In [ ]:
import importlib
from argparse import Namespace
import train_autoencoder
importlib.reload(train_autoencoder)
checkpoint_path = PROJECT_ROOT / "checkpoints/best_autoencoder.pth"
history_path = PROJECT_ROOT / "results/ae_training_history.csv"
if TRAIN_MODEL:
    assert DEVICE.type == "cuda", "Enable a Colab GPU before full training."
    training_summary = train_autoencoder.train(Namespace(
        splits_dir=PROJECT_ROOT/"data/splits", checkpoint_path=checkpoint_path, history_path=history_path,
        curve_path=PROJECT_ROOT/"outputs/ae/ae_training_curve.png", grid_path=PROJECT_ROOT/"outputs/ae/trained_reconstruction_grid.png",
        image_size=128, batch_size=32, num_workers=0, learning_rate=0.001, max_epochs=20, patience=4, seed=42, smoke_test=False))
else:
    assert checkpoint_path.is_file(), f"Missing trained checkpoint: {checkpoint_path}"
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    assert checkpoint["epoch"] == 20, f"Expected epoch-20 checkpoint, found epoch {checkpoint['epoch']}"
    model.load_state_dict(checkpoint["model_state_dict"]); model.eval()
    print("Loaded best checkpoint from epoch", checkpoint["epoch"])

## 10 — Training Results

In [ ]:
training_results = pd.DataFrame({"Result": ["Epochs completed", "First training loss", "Final training loss", "Best validation loss", "Best epoch", "Training GPU", "Training time"], "Value": [20, 0.0279316258, 0.0042993340, 0.0041614504, 20, "Tesla T4", "733.16 seconds"]})
display(training_results)
history = pd.read_csv(history_path) if history_path.is_file() else pd.DataFrame()
if len(history) == 20:
    ax = history.plot(x="epoch", y=["train_loss", "validation_loss"], marker="o", figsize=(8,5), title="Standard AE Training History")
    ax.set_ylabel("MSE loss"); ax.grid(alpha=.3); plt.show()
else:
    print(f"WARNING: Expected 20 history rows, found {len(history)}. Copy the GPU ae_training_history.csv into results/ to display the full curve.")

## 11 — Reconstruction Results

Examples below come only from the held-out test split.

In [ ]:
selected_images, selected_names = [], []
class_counts = {"authentic": 0, "tampered": 0}
for batch in loaders["test"]:
    for image, name in zip(batch["image"], batch["class_name"]):
        if class_counts[name] < 3:
            selected_images.append(image); selected_names.append(name); class_counts[name] += 1
    if all(count == 3 for count in class_counts.values()): break
test_inputs = torch.stack(selected_images).to(DEVICE)
with torch.no_grad(): test_outputs = model(test_inputs).cpu()
fig, axes = plt.subplots(len(selected_images), 2, figsize=(7, 3*len(selected_images)))
for row, (original, class_name) in enumerate(zip(selected_images, selected_names)):
    name = class_name.title()
    axes[row,0].imshow(original.permute(1,2,0)); axes[row,0].set_title(f"{name} — Original")
    axes[row,1].imshow(test_outputs[row].permute(1,2,0)); axes[row,1].set_title(f"{name} — Reconstructed")
    axes[row,0].axis("off"); axes[row,1].axis("off")
plt.tight_layout(); plt.show()

## 12 — Test Metrics

MSE is better when lower, PSNR is better when higher, and SSIM is better when closer to 1.

In [ ]:
test_metrics = json.loads((PROJECT_ROOT / "results/ae_test_metrics.json").read_text())
overall = test_metrics["overall"]
display(pd.DataFrame({"Metric": ["MSE", "PSNR (dB)", "SSIM"], "Test mean": [overall["mse_mean"], overall["psnr_mean"], overall["ssim_mean"]]}))

## 13 — Authentic vs Tampered Analysis

The Autoencoder showed slightly higher reconstruction error and lower structural similarity for tampered images. This is an exploratory observation only; the model was not trained or validated as a forgery classifier.

In [ ]:
comparison = pd.DataFrame([
    {"Class": name.title(), "MSE": test_metrics[name]["mse_mean"], "PSNR (dB)": test_metrics[name]["psnr_mean"], "SSIM": test_metrics[name]["ssim_mean"]}
    for name in ("authentic", "tampered")])
display(comparison)
fig, axes = plt.subplots(1, 2, figsize=(12,4))
comparison.plot.bar(x="Class", y="MSE", ax=axes[0], legend=False, color=["#2878B5", "#E07A1F"], title="Mean Reconstruction MSE")
comparison.plot.bar(x="Class", y="SSIM", ax=axes[1], legend=False, color=["#2878B5", "#E07A1F"], title="Mean Reconstruction SSIM")
plt.tight_layout(); plt.show()

## 14 — Denoising Autoencoder (Next Stage)

Reserved for the next experiment. It will add configurable Gaussian noise (`noise_std = 0.10`), use **noisy image → clean image** training pairs, retain the same architecture, and fine-tune from the standard AE checkpoint. Planned presentation: **Clean | Noisy | Denoised**, followed by denoising metrics. No denoising result is claimed yet.

## 15 — Final Comparison (Pending Denoising Experiment)

In [ ]:
display(pd.DataFrame([
    ["Standard AE", overall["mse_mean"], overall["psnr_mean"], overall["ssim_mean"], 24, 265571],
    ["Noisy input", "Pending", "Pending", "Pending", "—", "—"],
    ["Denoised output", "Pending", "Pending", "Pending", 24, 265571],
], columns=["Experiment", "MSE", "PSNR (dB)", "SSIM", "Compression ratio", "Parameters"]))

## 16 — Conclusion

- CASIA images were successfully reconstructed by the standard Autoencoder.
- The latent representation provides genuine **24× compression**.
- Test reconstruction quality was measured using MSE, PSNR, and SSIM.
- Tampered images showed slightly different reconstruction behavior in this exploratory comparison.
- The denoising experiment is the next planned stage and is not yet reported here.
- The model is not claimed to be a standalone forgery detector.